# Scraping Skytrax airline reviews

This is my scraper for the first module of the British Airways Data Science job simulation on Forage, which asked for customer reviews from Skytrax to analyse. The notebook reads the Skytrax A to Z list of airlines, collects each airline's overall category scores and up to 1,200 of its latest reviews, and writes them as CSV files under `../data/`.

The scraped reviews are not included in this repo. Outputs that printed rows of scraped data have been cleared, and the coverage figure is kept. If you run it, please check the site's terms of use first and keep the request rate low.

In [417]:
# import required libraries
import concurrent.futures
import json
import os
import random
import re
import string

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup

In [2]:
# define requests header
user_agents = [
    "Mozilla/5.0 (X11; CrOS x86_64 12871.102.0) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/81.0.4044.141 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_9_2) AppleWebKit/537.75.14 (KHTML, like Gecko) Version/7.0.3 Safari/537.75.14",
    "Mozilla/5.0 (compatible; MSIE 10.0; Windows NT 6.2; Win64; x64; Trident/6.0)",
    "Opera/9.25 (Windows NT 5.1; U; en)",
    "Mozilla/4.0 (compatible; MSIE 6.0; Windows NT 5.1; SV1; .NET CLR 1.1.4322; .NET CLR 2.0.50727)",
    "Mozilla/5.0 (compatible; Konqueror/3.5; Linux) KHTML/3.5.5 (like Gecko) (Kubuntu)",
    "Lynx/2.8.5rel.1 libwww-FM/2.14 SSL-MM/1.4.1 GNUTLS/1.2.9",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_11_2) AppleWebKit/601.3.9 (KHTML, like Gecko) Version/9.0.2 Safari/601.3.9",
]

In [3]:
HOME_URL = "https://www.airlinequality.com"

In [172]:
# obtain all airline names
all_airlines = []


response = requests.get(
    url=HOME_URL + "/review-pages/a-z-airline-reviews/",
    headers={"User-Agent": random.choice(user_agents)},
    timeout=5,
)
bs_object = BeautifulSoup(markup=response.text, features="html.parser")

# obtain airline names
airline_lists = ["a2z-ldr-" + i for i in string.ascii_uppercase]

for airline_list in airline_lists:
    airlines = bs_object.find("div", attrs={"id": airline_list}).findAll("a")
    for airline in airlines:
        all_airlines.append(airline.text)

---

In [174]:
def scrap_airline_overall_reviews(name):
    airline_name = name.replace(" ", "-")
    url = HOME_URL + "/airline-reviews/" + airline_name

    response = requests.get(
        url=url, headers={"User-Agent": random.choice(user_agents)}, timeout=5
    )

    bs_object = BeautifulSoup(markup=response.text, features="html.parser")

    ratings = bs_object.find(name="table", attrs={"class": "review-ratings"}).findAll(
        "tr"
    )

    scores = list()
    for rate in ratings:
        score = np.nan
        score_span = rate.findAll("td")[1].findAll(
            name="span", attrs={"class": "star fill"}
        )

        if score_span:
            score = score_span[-1].text

        scores.append(score)

    attributes = [
        "Food & Beverage",
        "Inflight Entertainment",
        "Seat Comfort",
        "Staff Service",
        "Value for Money",
    ]
    ratings = dict(zip(attributes, scores))
    ratings["airline"] = name

    review_scores.append(ratings)
    return

In [175]:
review_scores = list()

In [176]:
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(scrap_airline_overall_reviews, all_airlines)

In [193]:
review_scores_df = pd.DataFrame(review_scores)
review_scores_df = review_scores_df[
    [
        "airline",
        "Food & Beverage",
        "Inflight Entertainment",
        "Seat Comfort",
        "Staff Service",
        "Value for Money",
    ]
]
review_scores_df.rename(columns={"airline": "Airline"}, inplace=True)
review_scores_df

In [199]:
review_scores_df.fillna(-1, inplace=True)
review_scores_df.astype(
    {
        "Airline": str,
        "Food & Beverage": np.int8,
        "Inflight Entertainment": np.int8,
        "Seat Comfort": np.int8,
        "Staff Service": np.int8,
        "Value for Money": np.int8,
    }
)
review_scores_df

In [200]:
review_scores_df.to_csv("../data/review_scores.csv", index=False)

There are some missing airline data, this is caused by different airline urls and highly likely because there is an inconsistent in database. However, the script still manage to scrap 89% data from the skytrax website.

---

In [426]:
def obtain_review_information(review):
    data = dict()
    data["header"] = review.find("h2", {"class": "text_header"}).text
    data["rating"] = (
        review.find("span", {"itemprop": "ratingValue"}).text
        if review.find("span", {"itemprop": "ratingValue"})
        else None
    )
    data["review_date"] = re.search(
        pattern=r"\d{4}-\d{2}-\d{2}",
        string=str(review.find("meta", {"itemprop": "datePublished"})),
    ).group()
    data["comment"] = review.find("div", {"class": "text_content"}).text
    data["trip_verified"] = True if "✅ Trip Verified" in data["comment"] else False

    data["comment"] = data["comment"].replace("✅ Trip Verified | ", "").strip()

    data["aircraft"] = (
        review.find("td", {"class": "aircraft"}).findNext("td").text
        if review.find("td", {"class": "aircraft"})
        else None
    )
    data["type_of_traveller"] = (
        review.find("td", {"class": "type_of_traveller"}).findNext("td").text
        if review.find("td", {"class": "type_of_traveller"})
        else None
    )
    data["seat_type"] = (
        review.find("td", {"class": "cabin_flown"}).findNext("td").text
        if review.find("td", {"class": "cabin_flown"})
        else None
    )
    data["route"] = (
        review.find("td", {"class": "route"}).findNext("td").text
        if review.find("td", {"class": "route"})
        else None
    )
    data["date_flown"] = (
        review.find("td", {"class": "date_flown"}).findNext("td").text
        if review.find("td", {"class": "date_flown"})
        else None
    )

    data["seat_comfort"] = None
    if review.find("td", {"class": "seat_comfort"}):
        if (
            review.find("td", {"class": "seat_comfort"})
            .findNext("td")
            .findAll("span", {"class": "star fill"})
        ):
            data["seat_comfort"] = (
                review.find("td", {"class": "seat_comfort"})
                .findNext("td")
                .findAll("span", {"class": "star fill"})[-1]
                .text
            )

    data["cabin_staff_service"] = None
    if review.find("td", {"class": "cabin_staff_service"}):
        if (
            review.find("td", {"class": "cabin_staff_service"})
            .findNext("td")
            .findAll("span", {"class": "star fill"})
        ):
            data["cabin_staff_service"] = (
                review.find("td", {"class": "cabin_staff_service"})
                .findNext("td")
                .findAll("span", {"class": "star fill"})[-1]
                .text
            )

    data["food_and_beverages"] = None
    if review.find("td", {"class": "food_and_beverages"}):
        if (
            review.find("td", {"class": "food_and_beverages"})
            .findNext("td")
            .findAll("span", {"class": "star fill"})
        ):
            data["food_and_beverages"] = (
                review.find("td", {"class": "food_and_beverages"})
                .findNext("td")
                .findAll("span", {"class": "star fill"})[-1]
                .text
            )

    data["food_and_beverages"] = None
    if review.find("td", {"class": "inflight_entertainment"}):
        if (
            review.find("td", {"class": "inflight_entertainment"})
            .findNext("td")
            .findAll("span", {"class": "star fill"})
        ):
            data["food_and_beverages"] = (
                review.find("td", {"class": "inflight_entertainment"})
                .findNext("td")
                .findAll("span", {"class": "star fill"})[-1]
            )

    data["ground_service"] = None
    if review.find("td", {"class": "ground_service"}):
        if (
            review.find("td", {"class": "ground_service"})
            .findNext("td")
            .findAll("span", {"class": "star fill"})
        ):
            data["ground_service"] = (
                review.find("td", {"class": "ground_service"})
                .findNext("td")
                .findAll("span", {"class": "star fill"})[-1]
                .text
            )

    data["value_for_money"] = None
    if review.find("td", {"class": "value_for_money"}):
        if (
            review.find("td", {"class": "value_for_money"})
            .findNext("td")
            .findAll("span", {"class": "star fill"})
        ):
            data["value_for_money"] = (
                review.find("td", {"class": "value_for_money"})
                .findNext("td")
                .findAll("span", {"class": "star fill"})[-1]
                .text
            )

    data["wifi_and_connecticity"] = None
    if review.find("td", {"class": "wifi_and_connectivity"}):
        if (
            review.find("td", {"class": "wifi_and_connectivity"})
            .findNext("td")
            .findAll("span", {"class": "star fill"})
        ):
            data["wifi_and_connectivity"] = (
                review.find("td", {"class": "wifi_and_connectivity"})
                .findNext("td")
                .findAll("span", {"class": "star fill"})[-1]
                .text
            )
    data["recommend"] = (
        review.find("td", {"class": "recommended"}).findNext("td").text == "yes"
        if review.find("td", {"class": "recommended"})
        else None
    )

    return data

In [427]:
def scrap_airline_reviews(name) -> pd.core.frame.DataFrame:
    airline_name = name.replace(" ", "-").lower()
    url = (
        HOME_URL
        + "/airline-reviews/"
        + airline_name
        + "/page/1/?sortby=post_date%3ADesc&pagesize=100"
    )
    response = requests.get(
        url=url, headers={"User-Agent": random.choice(user_agents)}, timeout=5
    )

    bs_object = BeautifulSoup(markup=response.text, features="html.parser")

    review_num = int(
        bs_object.find(name="span", attrs={"itemprop": "reviewCount"})
        .text.replace("\n", "")
        .replace("\t", "")
    )
    review_num = int(np.round(1270 / 100))

    all_reviews = list()
    for page_num in range(1, review_num):
        url = (
            HOME_URL
            + "/airline-reviews/"
            + airline_name
            + "/page/"
            + str(page_num)
            + "/?sortby=post_date%3ADesc&pagesize=100"
        )

        response = requests.get(
            url=url, headers={"User-Agent": random.choice(user_agents)}, timeout=5
        )
        bs_object = BeautifulSoup(markup=response.text, features="html.parser")
        reviews = bs_object.findAll(name="article", attrs={"itemprop": "review"})

        for review in reviews:
            data = obtain_review_information(review)
            all_reviews.append(data)

    df = pd.DataFrame(all_reviews)
    df.to_csv(
        f"../data/airline_reviews/{name.replace(' ', '-').lower()}.csv", index=False
    )
    return

In [429]:
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(scrap_airline_reviews, all_airlines)

In [430]:
data_path = [f for f in os.listdir("../data/airline_reviews") if "csv" in f]

In [432]:
len(data_path) / len(all_airlines)

0.8931860036832413

After scraping review data for each airlines, we are able to obtain 89% airline reviews.

---

Cleanning data

In [451]:
# combine all data into a single dataframe
review_df = pd.read_csv("../data/airline_reviews/" + data_path[0])
review_df["airline"] = data_path[0].replace(".csv", "")

for f in data_path[1::]:
    df = pd.read_csv("../data/airline_reviews/" + f)
    df["airline"] = f.replace(".csv", "")
    review_df = pd.concat([review_df, df], axis=0)

review_df

In [452]:
# remove "" wrapper in each header
review_df["header"] = review_df.header.str.replace('"', "").str.strip()

In [453]:
# remove unexpected values in comment
review_df["comment"] = review_df.comment.str.replace("Not Verified |", "").str.strip()
review_df["comment"] = review_df.comment.str.replace("|", "").str.strip()

In [505]:
def extract_number(x):
    res = re.search(r"\d", str(x))
    if res:
        return res.group()
    else:
        return np.nan

In [507]:
review_df["food_and_beverages"] = review_df.agg(
    lambda x: extract_number(x["food_and_beverages"]), axis=1
)

In [509]:
review_df.to_csv("../data/review_comments.csv", index=False)

---

In [510]:
# customer review ratings
url = "https://www.airlinequality.com/review-pages/a-z-airline-reviews/airline-review-ratings/"
response = requests.get(
    url=url, headers={"User-Agent": random.choice(user_agents)}, timeout=5
)

bs_object = BeautifulSoup(markup=response.text, features="html.parser")

In [553]:
review_table = bs_object.find("div", {"class": "tabs-content"})

In [556]:
scores = [
    i.strip()
    for i in str(BeautifulSoup(review_table.text, features="html.parser")).split("    ")
]
scores = [i.replace(" Airline Score (max 10) Reviews", "") for i in scores[1::]]

In [559]:
review_table = list()
for item in scores:
    temp = item.split(" ")
    airline, s, r = " ".join(temp[:-2]), temp[-2], temp[-1]
    review_table.append((airline, s, r))

In [560]:
review_ratings_df = pd.DataFrame(
    review_table, columns=["airline", "score (max 10)", "reviews"]
)

In [561]:
review_ratings_df

In [562]:
airline_links = bs_object.findAll(
    "div", {"class": "aggregateColumn aggregateColumn_two"}
)
airline_links = [
    str(l)
    .replace('<div class="aggregateColumn aggregateColumn_two"><a href="', "")
    .split(">")[0]
    for l in airline_links
]

In [563]:
airline_links = [
    HOME_URL + l
    for l in airline_links
    if '<div class="aggregateColumn aggregateColumn_two"' not in l
]

In [567]:
review_ratings_df["links"] = airline_links

In [568]:
review_ratings_df

In [569]:
review_ratings_df.to_csv("../data/review_ratings_df.csv", index=False)